# Tutorial: Building a Reflective Agentic Workflow for GDP Collection

### Learning Outcomes
1.  **Build** a multi-step agentic workflow using the **WAT (Workflow, Agents, Tools)** framework.
2.  **Apply** reflective design patterns (**Plan -> Act -> Evaluate -> Refine**) to improve data quality.
3.  **Implement** deterministic tools alongside probabilistic LLMs for reliable research.
4.  **Defend** workflow decisions based on **Source Tiering** and **Validation Gates**.

---

### Core Concepts: The Reflective Loop
An **Agent** is not just a call to an LLM. It is a system that observes its environment, performs an action, evaluates the result, and decides whether to refine its approach. 

#### Workflow Diagram

```mermaid
%%{init: {"flowchart": {"nodeSpacing": 18, "rankSpacing": 20, "diagramPadding": 4, "useMaxWidth": true}, "themeVariables": {"fontSize": "12px"}}}%%
graph LR
    A[Input City Country] --> B[Build Query Pack]
    B --> C[Primary Retrieval]
    C --> D{Intent Quota Met}
    D -- No --> E[Adaptive Expansion]
    D -- Yes --> F[Rank And Score]
    E --> F

    F --> G[Fetch And Cache]
    G --> H{Research Mode Enabled}

    H -- Yes --> I{City Mention Found}
    I -- Yes --> J[LLM First Extraction]
    I -- No --> K[Parser Extraction]
    J --> L{LLM Fact Found}
    L -- No --> K

    H -- No --> K
    K --> M{Parser Fact Found}
    M -- No --> N{Fallback Allowed And Budget}
    N -- Yes --> O[LLM Fallback Extraction]
    N -- No --> X[Reject Candidate]

    L -- Yes --> P[Normalize And FX]
    M -- Yes --> P
    O --> P

    P --> Q{Validation Gates Pass}
    Q -- Yes --> R[Accept Candidate]
    Q -- No --> X

    R --> S{Valid Row Exists}
    S -- Yes --> T[Select Final Row]
    S -- No --> U[Downscaled Fallback InReview]
    U --> T

    X --> V[Log Failure Trace]
    V --> S
```

> **Note:** To render the diagram in VS Code or Jupyter, ensure you have a Mermaid extension installed, or view it as a markdown preview.

#### Diagram Legend
- **Intent Quota Met** means: in the current candidate pool, we found at least `urls_per_city_for_extraction` intent-valid candidates (default 5) while scanning up to `max_urls_to_try_per_city` URLs (default 20), with at least 2 distinct domains when possible.
- A candidate is **intent-valid** only if it passes all of these checks:
  - country consistency check
  - not country-level-only page
  - city GDP intent check (city mention + GDP-level signal; noisy topics like throughput/rent/house-price are rejected)
- **Validation Gates Pass** means `EvaluatorAgent` accepts the normalized fact after checks on:
  - geography consistency and evidence presence
  - metric type (`level` only; reject growth/share/per-capita)
  - year bounds
  - value bounds and GDP-per-capita plausibility
  - FX completeness for non-USD values
  - source quality fields
  - hallucination audit for LLM-assisted rows



In [ ]:
# For google colab set up
!git clone https://github.com/brookefzy/agents-for-urban-planning.git /content/agents-for-urban-planning
%cd /content/agents-for-urban-planning/tutorials/01_city_gdp_collection
!pip install -r requirements.txt

## Step 0: API Configuration & Safety Safeguards

To run this research agent, you need access to search and reasoning APIs. 

### 1. API Keys
- **Tavily:** [Get a free key here](https://tavily.com/) for web search (1,000 free searches/month).
- **OpenAI:** [Get an API key here](https://platform.openai.com/) for the LLM reflection layer.

### 2. Setting up your Environment
**Local (VS Code/Desktop):** Create a file named `.env` in this folder and add:
```env
OPENAI_API_KEY=sk-...
TAVILY_API_KEY=tvly-...
```

**Google Colab:** Click the **Secrets (Key Icon)** on the left sidebar and add `OPENAI_API_KEY` and `TAVILY_API_KEY` there.

> **⚠️ BUDGET SAFEGUARDS:** This tutorial is hard-coded with `SAFEGUARD_LIMITS` to ensure you don't accidentally spend more than a few cents. We limit the number of cities, search results, and LLM calls.

## Step 1: Configuration & Environment Setup

In [ ]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv

# --- TUTORIAL SAFEGUARDS ---
SAFEGUARD_MAX_CITIES = 3      # Never process more than 5 cities at once
SAFEGUARD_MAX_SEARCH_K = 5    # Never fetch more than 3 URLs per city
SAFEGUARD_MAX_LLM_CALLS = 30  # Maximum total LLM extractions allowed
# ---------------------------

# Ensure the current folder is in the system path to import local agents
module_path = Path(".").resolve()
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))

# Load API Keys
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    os.environ["TAVILY_API_KEY"] = userdata.get('TAVILY_API_KEY')
    print("Using Google Colab Secrets.")
except ImportError:
    load_dotenv("../.env")
    print("Using local .env file.")

if not os.getenv("TAVILY_API_KEY"):
    print("Error: TAVILY_API_KEY not found. Search will fail.")

print(f"Environment Ready. Working in: {module_path}")


Using local .env file.
Environment Ready. Working in: /Users/yuan/Documents/GitHub/tutorials/agents-for-urban-planning/tutorials/01_city_gdp_collection


## Step 2: Defining the Task Schema & Agents
We will use three specialized agents from the `agents/` folder:
1.  **SearchAgent:** Responsible for "Query Expansion" and retrieval.
2.  **ExtractorAgent:** Responsible for parsing HTML/Text into structured data.
3.  **EvaluatorAgent:** The "Judge" that validates the data against scientific gates (Z-scores, year range, source quality).

In [2]:
from pathlib import Path
from agents.search import SearchAgent
from agents.extractor import ExtractorAgent
from agents.evaluator import EvaluatorAgent
from agents.normalizer import NormalizerAgent
from tools.http_fetch import fetch_with_cache

# Initialize our Agentic Team with Safeguards
searcher = SearchAgent(top_k=SAFEGUARD_MAX_SEARCH_K, search_engine="tavily")
extractor = ExtractorAgent()
evaluator = EvaluatorAgent()
normalizer = NormalizerAgent()

# Notebook runtime options
# If True: attempt LLM extraction first (research mode) when city appears in content.
# If False: keep deterministic parser as primary path and use LLM only as fallback.
TUTORIAL_LLM_RESEARCH_MODE = True

# Shared fetch cache path for reproducibility in notebook execution
FETCH_CACHE_DIR = Path('.tmp/tutorial_notebook_fetch_cache')
FETCH_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Global counter for LLM calls to prevent cost overruns
llm_call_counter = 0


## Step 3: The Reflective Workflow (Plan -> Act -> Evaluate -> Refine)

Instead of just asking an LLM for the answer, we build a **Reflective Loop**:
1.  **Plan:** Build search queries for the city.
2.  **Act (Search):** Get URLs and snippets.
3.  **Act (Extract):**
    - **Default mode:** Deterministic parsing first (fast, cheap, auditable), then LLM fallback if needed.
    - **LLM research mode (`TUTORIAL_LLM_RESEARCH_MODE=True`):** LLM extraction first when city name appears in content, parser as fallback if needed.
4.  **Evaluate:** Run extraction through **Validation Gates**.
5.  **Refine:** Retry with alternate method if data is missing/invalid.

> Notebook safeguard: after retrieval, we enforce a hard cap of `SAFEGUARD_MAX_SEARCH_K` candidate URLs per city before extraction.


In [ ]:
def process_city_reflectively(city, country, llm_research_mode=TUTORIAL_LLM_RESEARCH_MODE):
    global llm_call_counter
    print(f"\n>>> Researching: {city}, {country} | llm_research_mode={llm_research_mode}")

    # 1. PLAN & ACT: Search (Limited by Safeguard)
    candidates = searcher.search_city(city, country, year=2024)
    # IMPORTANT: SearchAgent top_k is applied per query variant, so we enforce
    # an explicit per-city cap here for tutorial cost/scope control.
    total_before_cap = len(candidates)
    candidates = [c for c in candidates if c.get('source_url')][:SAFEGUARD_MAX_SEARCH_K]
    print(f"  Candidates returned: {total_before_cap}; processing first {len(candidates)} due to SAFEGUARD_MAX_SEARCH_K={SAFEGUARD_MAX_SEARCH_K}")

    results = []
    for cand in candidates:
        url = cand['source_url']
        print(f"  Fetching: {url}")

        # Fetch content (cached for reproducibility)
        try:
            fetched = fetch_with_cache(url, cache_dir=FETCH_CACHE_DIR)
            content = fetched.get("content_text", "")
        except Exception:
            content = cand.get("snippet", "")

        city_in_content = city.lower() in (content or '').lower()
        facts = []
        method = "direct_parser"
        reasons = []

        # 2. ACT: Choose extraction strategy
        if llm_research_mode and city_in_content and llm_call_counter < SAFEGUARD_MAX_LLM_CALLS:
            llm_call_counter += 1
            facts = extractor.extract_with_llm(city, country, content)
            method = "llm_research_agent"
            if not facts:
                # Parser fallback keeps the tutorial robust when LLM returns no usable fact.
                facts = extractor.extract(city, country, content)
                if facts:
                    method = "llm_research_parser_fallback"
        else:
            # Default baseline: deterministic first
            facts = extractor.extract(city, country, content)
            method = "direct_parser"
            if (not facts) and city_in_content and llm_call_counter < SAFEGUARD_MAX_LLM_CALLS:
                print("    [!] Deterministic parse produced no valid fact. Trying LLM fallback...")
                llm_call_counter += 1
                facts = extractor.extract_with_llm(city, country, content)
                if facts:
                    method = "llm_fallback"

        # 3. EVALUATE
        is_valid = False
        if facts:
            fact_dict = {**vars(facts[0]), **cand, "method": method}
            fact_dict = normalizer.normalize_candidate(fact_dict)
            fact_dict["_source_content"] = content
            is_valid, reasons = evaluator.evaluate_candidate(fact_dict)

        # 4. REFINE / decide
        if is_valid:
            usd_val = fact_dict.get('gdp_usd')
            yr_val = fact_dict.get('year')
            if usd_val is not None:
                print(f"    [✓] Success via {method}: {usd_val:,.0f} USD ({yr_val})")
            else:
                print(f"    [✓] Success via {method}: year={yr_val}")
            results.append(fact_dict)
            break
        else:
            why = reasons if reasons else ("No facts found" if not facts else "Validation failed")
            print(f"    [x] Candidate rejected ({method}): {why}")

        if llm_call_counter >= SAFEGUARD_MAX_LLM_CALLS:
            print("    [!] LLM budget reached for this tutorial run.")

    return results


## Step 3A: Inspect the Search Plan and Candidate URLs (8-10 minutes)
Before running a batch, inspect what the retrieval agent is doing for **one city**.

### Student task
1. Run the cell below.
2. Review the candidate URLs/snippets.
3. Mark which source you would trust first (and why).
4. Discuss: Which candidates look risky even before extraction?


In [ ]:
import pandas as pd

inspect_city, inspect_country = "Zurich", "Switzerland"
print(f"Inspecting retrieval for: {inspect_city}, {inspect_country}")

try:
    debug_candidates = searcher.search_city(inspect_city, inspect_country, year=2024)
    print(f"Retrieved {len(debug_candidates)} candidate URLs.")
except Exception as e:
    debug_candidates = []
    print(f"Search failed: {type(e).__name__}: {e}")

if debug_candidates:
    debug_df = pd.DataFrame(debug_candidates)
    preferred_cols = [
        "source_url", "source_domain", "source_tier_label", "query", "snippet"
    ]
    visible_cols = [c for c in preferred_cols if c in debug_df.columns]
    display(debug_df[visible_cols])
else:
    print("No candidates returned. Check API keys / network access and continue to the batch section using a demo discussion.")


## Step 3B: Inspect One Candidate Through the Validation Gates (8-10 minutes)
Now trace a **single URL** through extraction and evaluation to see where failures happen.

### Student task
- Run the cell below.
- Read the extraction preview and validation result.
- Identify which gate(s) failed and what you would change (search query, parser, or validation rule).


In [ ]:
debug_content = ""
debug_candidate = None
debug_fact = None

if not globals().get("debug_candidates"):
    print("No debug candidates available from Step 3A. Run Step 3A first or skip to the batch activity.")
else:
    debug_candidate = debug_candidates[0]
    debug_url = debug_candidate.get("source_url")
    print(f"Tracing first candidate: {debug_url}")

    try:
        fetched = fetch_with_cache(debug_url, cache_dir=FETCH_CACHE_DIR)
        debug_content = fetched.get("content_text", "") or ""
        print(f"Fetched content length: {len(debug_content):,} characters")
    except Exception as e:
        debug_content = debug_candidate.get("snippet", "") or ""
        print(f"Fetch failed ({type(e).__name__}); falling back to snippet text.")

    preview = (debug_content[:600] + "...") if len(debug_content) > 600 else debug_content
    print("\n--- Content Preview ---")
    print(preview if preview else "[No content preview available]")

    facts = extractor.extract(inspect_city, inspect_country, debug_content)
    print(f"\nDeterministic extractor returned {len(facts) if facts else 0} fact(s).")

    if facts:
        debug_fact = {**vars(facts[0]), **debug_candidate, "city": inspect_city, "country": inspect_country, "method": "direct_parser"}
        debug_fact = normalizer.normalize_candidate(debug_fact)
        debug_fact["_source_content"] = debug_content
        ok, reasons = evaluator.evaluate_candidate(debug_fact)

        print("\n--- First Extracted Candidate (normalized) ---")
        keys_to_show = [
            "city", "country", "year", "gdp_raw", "currency", "gdp_usd",
            "geo_level", "source_tier_label", "method", "status", "evidence_text"
        ]
        for k in keys_to_show:
            if k in debug_fact:
                v = debug_fact.get(k)
                if isinstance(v, str) and len(v) > 180:
                    v = v[:180] + "..."
                print(f"{k}: {v}")

        print("\nValidation result:", "PASS" if ok else "FAIL")
        print("Reasons:", reasons if reasons else "None")
    else:
        print("No deterministic extraction found. This is exactly when the reflective fallback becomes useful.")


## Step 4A: Activity - Collecting Data for a Batch of Cities
We will run a sample batch and observe how the **Evaluator** prevents bad data from reaching the final table. Note that we respect the `SAFEGUARD_MAX_CITIES` limit.


In [ ]:
import pandas as pd

sample_cities = [
    ("Zurich", "Switzerland"),
    ("Nairobi", "Kenya"),
    ("Jakarta", "Indonesia"), # This will be sliced off by safeguards
]

# Apply Safeguard Slicing
active_batch = sample_cities[:SAFEGUARD_MAX_CITIES]
print(f"Running batch of {len(active_batch)} cities (Safeguard Active).")
print(f"LLM research mode: {TUTORIAL_LLM_RESEARCH_MODE}")

final_data = []
for city, country in active_batch:
    res = process_city_reflectively(city, country, llm_research_mode=TUTORIAL_LLM_RESEARCH_MODE)
    if res:
        final_data.extend(res)

df = pd.DataFrame(final_data)
if not df.empty:
    display(df[['city', 'country', 'year', 'gdp_usd', 'source_tier_label', 'method', 'status']])
else:
    print("\nNo data collected. Verify your API keys and check the console logs for failures.")


## Step 4B: Batch Diagnostics and Quality Review (8-10 minutes)
A successful run is not enough. Inspect **how** the workflow succeeded.

### Student task
- Run the diagnostics cell.
- Identify rows that require manual review.
- Compare `method` and `source_tier_label` distributions.


In [ ]:
print(f"LLM fallback calls used in this session: {llm_call_counter}/{SAFEGUARD_MAX_LLM_CALLS}")

if df.empty:
    print("No rows to diagnose yet.")
else:
    print(f"Rows collected: {len(df)}")

    if "status" in df.columns:
        print("\nStatus counts:")
        display(df["status"].value_counts(dropna=False).rename_axis("status").to_frame("rows"))

    if "method" in df.columns:
        print("\nMethod counts:")
        display(df["method"].value_counts(dropna=False).rename_axis("method").to_frame("rows"))

    if "source_tier_label" in df.columns:
        print("\nSource tier counts:")
        display(df["source_tier_label"].value_counts(dropna=False).rename_axis("source_tier_label").to_frame("rows"))

    review_mask = False
    if "status" in df.columns:
        review_mask = review_mask | df["status"].astype(str).str.lower().eq("inreview")
    if "method" in df.columns:
        review_mask = review_mask | df["method"].astype(str).str.lower().isin(["llm_fallback", "llm_research_agent", "llm_research_parser_fallback"])

    review_cols = [
        "city", "country", "year", "gdp_usd", "method", "status",
        "source_tier_label", "source_url", "failure_reasons"
    ]
    visible_review_cols = [c for c in review_cols if c in df.columns]

    review_df = df.loc[review_mask, visible_review_cols] if hasattr(review_mask, 'any') and review_mask.any() else pd.DataFrame(columns=visible_review_cols)
    print("\nRows to inspect manually (LLM-assisted methods or inReview):")
    if review_df.empty:
        print("None in this batch.")
    else:
        display(review_df)


## Step 4C: Validation Gate Experiments (10 minutes)
Use one real row and test how the evaluator reacts when you introduce common research-quality problems.

### Student task
- Run the experiment cell.
- Compare which errors are caught automatically.
- Discuss one risk the current evaluator **does not** catch yet.


In [ ]:
if df.empty:
    print("No rows available. Run Step 4A first.")
else:
    base = df.iloc[0].to_dict()
    # Force deterministic mode for this classroom experiment so we focus on validation gates,
    # not the extra hallucination audit used for llm_fallback rows.
    base["method"] = "direct_parser"
    base["_source_content"] = f"{base.get('city', '')} {base.get('evidence_text', '')}"

    experiments = [
        ("Baseline (expected pass or near-pass)", {}),
        ("Future year", {"year": 2035}),
        ("Implausibly high GDP", {"gdp_usd": 1e16}),
        ("Missing evidence text", {"evidence_text": ""}),
    ]

    rows = []
    for name, patch in experiments:
        cand = dict(base)
        cand.update(patch)
        ok, reasons = evaluator.evaluate_candidate(cand)
        rows.append({
            "experiment": name,
            "pass": ok,
            "reasons": ", ".join(reasons) if reasons else "None",
        })

    exp_df = pd.DataFrame(rows)
    display(exp_df)

    print("\nReflection prompt: Which missing rule would you add for your own research domain?")


## Step 4D: Optional Extensions (5-10 minutes, if time allows)
Choose **one** experiment and compare results.

1. Increase `SAFEGUARD_MAX_SEARCH_K` from `3` to `5` and rerun the batch. Did quality improve?
2. Set `SAFEGUARD_MAX_LLM_CALLS = 0` and rerun. What does the workflow lose?
3. Replace one city in `sample_cities` with a harder case (smaller city / ambiguous name).
4. Compare two rows and decide which one you would trust in a planning memo.

Tip: If you change safeguards, rerun from the agent initialization cell onward (Step 2 onward is safest).


## Step 5: Reproducibility & Research Rigor
Data without provenance is just a number. Our workflow ensures reproducibility by:
1.  **Source Tiering:** Explicitly labeling if data came from an Official Bureau (Tier 1) vs Wikipedia (Tier 3).
2.  **Evidence Trails:** Every row contains `evidence_text` and an `evidence_path` (like a table index or character range).
3.  **Confidence & Uncertainty:** Marking rows as `inReview` if they used a fallback method.

In [ ]:
if not df.empty:
    # Export for scientific analysis
    df.to_csv("tutorial_gdp_results.csv", index=False)
    print("Artifact Saved: tutorial_gdp_results.csv")


### Conclusion
You have just built a **Production-Grade Agentic Workflow**. 

**Key Takeaways:**
- **Agents are Iterative:** We didn't stop at the first failure; we reflected and used a different tool (LLM).
- **Validation is Mandatory:** The `EvaluatorAgent` acts as a guardrail against LLM hallucinations.
- **Reproducibility is the Goal:** By saving URLs, evidence, and source tiers, anyone can verify our research.

## Lessons Learned from a Real Debugging Session (March 2, 2026)

Today's live testing surfaced several non-obvious failure modes that are common in real-world agentic data workflows:

1. **Recall quality comes before extraction quality.**
   If search retrieves irrelevant pages (throughput, rent, house price, country GDP), better parsing or stronger LLM prompts will not rescue the pipeline.

2. **"Top-k URLs" is not the same as "k useful candidates."**
   We changed the workflow from processing the first 5 URLs to scanning up to 20 and selecting up to 5 intent-valid candidates.

3. **Guardrails should be explicit and observable.**
   New fields like `failure_reasons`, `prefetch_confidence`, `prefetch_reasons`, and `quota_stage` make debugging faster and auditable.

4. **LLM usage needs precise semantics.**
   `llm_attempted`, `llm_used`, and `llm_error` should be consistent with logs, otherwise users cannot trust diagnostics.

5. **Metric/type confusion is a major hidden risk.**
   Growth rates, index values, or year tokens can be mistaken as GDP levels. We enforced stricter metric-type normalization and validation.

6. **Network instability can dominate outcomes.**
   DNS and fetch failures caused many false negatives. We added retry and host-variant fallback (`www` <-> root domain), but logs still matter for separating data issues from transport issues.

7. **Avoid overfitting to demo cities or fixed trusted lists.**
   Hard-coded trusted domains are convenient for demos but do not scale globally. Better patterns are adaptive query expansion, confidence scoring, and failure-driven re-query loops.

8. **Fallback is useful, but should be clearly labeled.**
   Downscaled fallback keeps the pipeline resilient, but outputs should remain `inReview` and never be confused with extracted city-level GDP facts.

### Reflection prompts for students
- If your pipeline returns no results for a known city, how would you determine whether the root cause is retrieval, extraction, normalization, or network?
- What should be considered a hard fail vs an acceptable fallback in an urban-planning decision context?
- How would you design evaluation metrics that reward both correctness and explainability?
